In [ ]:
# =========================================================
# LCBD attribution (NO period split)
# Clean output: Overall marginal effects only
# =========================================================

library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(car)
library(MuMIn)
library(marginaleffects)
library(tibble)

# =========================
# 1) Read data
# =========================
df <- read_csv("D:/NC/Data/rivernet/inputdata/Richness_slope.csv")

df <- df %>%
  mutate(
    SiteID   = as.character(SiteID),
    HYBAS_ID = as.character(HYBAS_ID),
    Protocol = as.character(Protocol),
    zone     = as.character(zone)
  )

# =========================
# 2) Numeric columns
# =========================
num_cols <- c(
  "slope_LCBD",
  "mean_temp","slope_temp","mean_flow",
  "mean_organic","mean_salinity",
  "HFP_mean","elevation"
)

df[num_cols] <- lapply(df[num_cols], as.numeric)

df <- df %>%
  filter(
    complete.cases(
      slope_LCBD,
      mean_temp, slope_temp, mean_flow,
      mean_organic, mean_salinity,
      HFP_mean, elevation,
      Protocol, zone, HYBAS_ID
    )
  )

# =========================
# 3) Z-score predictors
# =========================
vars <- c(
  "mean_temp","slope_temp","mean_flow",
  "mean_organic","mean_salinity",
  "HFP_mean","elevation"
)

for (v in vars) {
  df[[paste0(v, "_z")]] <- scale(df[[v]])[, 1]
}

# =========================
# 4) Model structure (NO period terms)
# =========================
fixed_form <- slope_LCBD ~
  mean_temp_z + slope_temp_z + mean_flow_z +
  mean_organic_z + mean_salinity_z +
  HFP_mean_z + elevation_z

form <- update(
  fixed_form,
  . ~ . + (1 | HYBAS_ID) + (1 | Protocol)
)

# =========================
# 5) VIF helpers
# =========================
zones <- c("All", "Cold", "Intermediate", "Warm")
vif_threshold <- 10
core_vars <- c("mean_temp", "slope_temp")  # never flag these

get_vif_vals <- function(dat) {
  lm_fit <- lm(fixed_form, data = dat)
  vif_raw <- car::vif(lm_fit)
  if (is.matrix(vif_raw)) {
    vif_raw[, "GVIF^(1/(2*Df))"]
  } else {
    vif_raw
  }
}

is_high_vif <- function(v, vif_vals) {
  if (v %in% core_vars) return(FALSE)
  term <- paste0(v, "_z")
  if (!(term %in% names(vif_vals))) return(FALSE)
  isTRUE(vif_vals[term] >= vif_threshold)
}

sig_label <- function(p) {
  if (is.na(p)) "" else if (p < 0.001) "***"
  else if (p < 0.01) "**"
  else if (p < 0.05) "*"
  else ""
}

# =========================
# 6) Run by zone
# =========================
effects_all <- list()
model_stats <- list()
vif_tables  <- list()

for (z in zones) {

  dat <- if (z == "All") df else filter(df, zone == z)
  if (nrow(dat) < 50) next

  cat("Running zone:", z, "\n")

  # -------- VIF --------
  vif_vals <- get_vif_vals(dat)
  vif_tables[[z]] <- data.frame(
    Zone = z,
    Term = names(vif_vals),
    VIF  = as.numeric(vif_vals),
    row.names = NULL
  )

  high_vif <- sapply(vars, is_high_vif, vif_vals = vif_vals)
  names(high_vif) <- vars

  # -------- Model --------
  m <- lmer(
    form, data = dat, REML = TRUE,
    control = lmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
  )

  # -------- Save summary --------
  writeLines(
    capture.output(summary(m)),
    paste0("C:/Users/Lenovo/Modelling of uniqueness trends/summary_LCBD_", z, ".txt")
  )

  # -------- Save fixed effects table --------
  coef_tab <- coef(summary(m)) %>%
    as.data.frame() %>%
    rownames_to_column("Term") %>%
    mutate(Zone = z)

  write_csv(
    coef_tab,
    paste0("C:/Users/Lenovo/Modelling of uniqueness trends/coef_LCBD_", z, ".csv")
  )

  # -------- Save random effects --------
  vc_tab <- as.data.frame(VarCorr(m)) %>%
    mutate(Zone = z)

  write_csv(
    vc_tab,
    paste0("C:/Users/Lenovo/Modelling of uniqueness trends/varcomp_LCBD_", z, ".csv")
  )

  # -------- Fit statistics --------
  r2 <- suppressWarnings(MuMIn::r.squaredGLMM(m))
  model_stats[[z]] <- data.frame(
    Zone = z,
    AIC = AIC(m),
    BIC = BIC(m),
    R2_marginal = r2[1],
    R2_conditional = r2[2],
    row.names = NULL
  )

  # -------- Marginal effects (overall only) --------
  for (v in vars) {

    if (isTRUE(high_vif[[v]])) {
      effects_all[[length(effects_all) + 1]] <- data.frame(
        Zone = z,
        Factor = v,
        Effect = NA,
        CI95_L = NA,
        CI95_U = NA,
        p = NA,
        sig = "",
        row.names = NULL
      )
      next
    }

    vz <- paste0(v, "_z")

    ame <- avg_slopes(
      m, variables = vz,
      newdata = dat
    )

    b  <- ame$estimate[1]
    ciL <- ame$conf.low[1]
    ciU <- ame$conf.high[1]
    p  <- ame$p.value[1]

    effects_all[[length(effects_all) + 1]] <- data.frame(
      Zone = z,
      Factor = v,
      Effect = b,
      CI95_L = ciL,
      CI95_U = ciU,
      p = p,
      sig = sig_label(p),
      row.names = NULL
    )
  }
}

# =========================
# 7) Output ((Dataset S4))
# =========================
write_csv(bind_rows(effects_all),
          "C:/Users/Lenovo/Modelling of uniqueness trends/LCBD_MarginalEffects_by_zone_OVERALL.csv")

write_csv(bind_rows(model_stats),
          "C:/Users/Lenovo/Modelling of uniqueness trends/Model_fit_stats_LCBD.csv")

Warning message:
"package 'lme4' was built under R version 4.5.2"
Loading required package: Matrix

Warning message:
"package 'lmerTest' was built under R version 4.5.2"

Attaching package: 'lmerTest'


The following object is masked from 'package:lme4':

    lmer


The following object is masked from 'package:stats':

    step


Warning message:
"package 'dplyr' was built under R version 4.5.2"

Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Warning message:
"package 'readr' was built under R version 4.5.2"
Warning message:
"package 'car' was built under R version 4.5.2"
Loading required package: carData

Warning message:
"package 'carData' was built under R version 4.5.2"

Attaching package: 'car'


The following object is masked from 'package:dplyr':

    recode


Warning message:
"package 'MuMIn' was built under R version 4.5.2"
Wa

Running zone: All 


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning messa

Running zone: Cold 


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning messa

Running zone: Intermediate 


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning messa

Running zone: Warm 


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning messa

🎉 完成：LCBD mixed models + VIF + avg_slopes(Overall only) + summaries + csv outputs


In [ ]:
library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(ggplot2)
library(patchwork)

# =========================
# 1) Read & prepare data
# =========================
df <- read_csv("D:/NC/Data/rivernet/inputdata/Richness_slope.csv")

df <- df %>%
  mutate(
    SiteID   = as.character(SiteID),
    HYBAS_ID = as.character(HYBAS_ID),
    Protocol = as.character(Protocol),
    zone     = as.character(zone)
  )

num_cols <- c(
  "slope_LCBD",
  "mean_temp","slope_temp","mean_flow",
  "mean_organic","mean_salinity",
  "HFP_mean","elevation"
)

df[num_cols] <- lapply(df[num_cols], as.numeric)

df <- df %>%
  filter(
    complete.cases(
      slope_LCBD,
      mean_temp, slope_temp, mean_flow,
      mean_organic, mean_salinity,
      HFP_mean, elevation,
      Protocol, zone, HYBAS_ID
    )
  )

vars <- c(
  "mean_temp","slope_temp","mean_flow",
  "mean_organic","mean_salinity",
  "HFP_mean","elevation"
)

for (v in vars) {
  df[[paste0(v, "_z")]] <- scale(df[[v]])[, 1]
}

# =========================
# 2) Model formula
# =========================
form <- slope_LCBD ~
  mean_temp_z + slope_temp_z + mean_flow_z +
  mean_organic_z + mean_salinity_z +
  HFP_mean_z + elevation_z +
  (1 | HYBAS_ID) + (1 | Protocol)

zones <- c("All", "Cold", "Intermediate", "Warm")

plot_list <- list()

# =========================
# 3) Fit & collect diagnostics
# =========================
for (z in zones) {

  dat <- if (z == "All") df else filter(df, zone == z)
  if (nrow(dat) < 50) next

  m <- lmer(
    form, data = dat, REML = TRUE,
    control = lmerControl(
      optimizer = "bobyqa",
      optCtrl = list(maxfun = 2e5)
    )
  )

  diag_df <- data.frame(
    fitted = fitted(m),
    resid  = resid(m)
  )

  # ---- Residuals vs Fitted ----
  p1 <- ggplot(diag_df, aes(fitted, resid)) +
    geom_point(size = 0.8, alpha = 0.7) +
    geom_hline(yintercept = 0, linetype = "dashed") +
    labs(
      title = paste(z, ": Residuals vs Fitted"),
      x = "Fitted values",
      y = "Residuals"
    ) +
    theme_bw(base_size = 11)

  # ---- Normal QQ ----
  p2 <- ggplot(diag_df, aes(sample = resid)) +
    stat_qq(size = 0.8, alpha = 0.8) +
    stat_qq_line(color = "red", linewidth = 0.8) +
    labs(
      title = paste(z, ": Normal Q–Q"),
      x = "Theoretical quantiles",
      y = "Sample quantiles"
    ) +
    theme_bw(base_size = 11)

  plot_list[[z]] <- p1 + p2
}

# =========================
# 4) Combine & save ONE figure
# =========================
final_plot <- wrap_plots(plot_list, ncol = 1)

ggsave(
  filename = "C:/Users/Lenovo/Modelling of uniqueness trends/S3.png",
  plot = final_plot,
  width = 10,
  height = 14,
  dpi = 400
)


Warning message:
"package 'patchwork' was built under R version 4.5.2"
Rows: 3417 Columns: 34
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (7): SiteID, Protocol, FlowClass, elev_class, HFP_class, geometry, zone
dbl (27): start_year, end_year, n_obs, slope_richness, p_richness, slope_tem...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


✅ LCBD residual diagnostics saved as ONE figure


In [ ]:
library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(spdep)

# =========================
# 1) Read data
# =========================
df <- read_csv(
  "D:/NC/Data/rivernet/inputdata/Richness_slope.csv"
)

df <- df %>%
  mutate(
    SiteID   = as.character(SiteID),
    HYBAS_ID = as.character(HYBAS_ID),
    Protocol = as.character(Protocol)
  )

# =========================
# 2) Scale predictors
# =========================
scale_cols <- c(
  "mean_temp","slope_temp","mean_flow",
  "mean_organic","mean_salinity",
  "HFP_mean","elevation"
)

for (v in scale_cols) {
  df[[paste0(v, "_z")]] <- scale(df[[v]])[, 1]
}

# =========================
# 3) LCBD mixed model
# =========================
form <- slope_LCBD ~
  mean_temp_z + slope_temp_z + mean_flow_z +
  mean_organic_z + mean_salinity_z +
  HFP_mean_z + elevation_z +
  (1 | HYBAS_ID) + (1 | Protocol) 

zones <- c("All","Cold","Intermediate","Warm")

out <- list()

# =========================
# 4) Moran’s I by zone
# =========================
for (z in zones) {

  dat <- if (z == "All") df else filter(df, zone == z)
  if (nrow(dat) < 50) next

  cat("Running Moran's I for LCBD - zone:", z, "\n")

  m <- lmer(form, data = dat, REML = TRUE)

  # ---- residuals used by model ----
  mf <- model.frame(m)
  mf$resid <- resid(m)

  dat_used <- dat[as.numeric(rownames(mf)), ]

  mf$SiteID    <- dat_used$SiteID
  mf$Longitude <- dat_used$Longitude
  mf$Latitude  <- dat_used$Latitude

  # ---- aggregate to SITE level ----
  site_res <- mf %>%
    group_by(SiteID) %>%
    summarise(
      resid = mean(resid, na.rm = TRUE),
      Longitude = first(Longitude),
      Latitude  = first(Latitude),
      .groups = "drop"
    )

  coords <- as.matrix(site_res[, c("Longitude", "Latitude")])

  # ---- spatial weights (same as richness) ----
  knn <- knearneigh(coords, k = 10)
  nb  <- knn2nb(knn)
  lw  <- nb2listw(nb, style = "W", zero.policy = TRUE)

  mi <- moran.test(
    site_res$resid,
    lw,
    randomisation = TRUE,
    zero.policy = TRUE
  )

  out[[z]] <- data.frame(
    Zone     = z,
    N_sites  = nrow(site_res),
    Moran_I  = unname(mi$estimate["Moran I statistic"]),
    P_value  = mi$p.value,
    row.names = NULL
  )
}

# =========================
# 5) Output ((Dataset S4))
# =========================
df_moran <- bind_rows(out)

print(df_moran)

write_csv(
  df_moran,
  "C:/Users/Lenovo/Modelling of uniqueness trends/MoranI_residuals_by_zone_SITE_LCBD.csv"
)

cat("🎉 Moran’s I (site-level residuals) for LCBD completed\n")


Warning message:
"package 'lme4' was built under R version 4.5.2"
Loading required package: Matrix

Warning message:
"package 'lmerTest' was built under R version 4.5.2"

Attaching package: 'lmerTest'


The following object is masked from 'package:lme4':

    lmer


The following object is masked from 'package:stats':

    step


Warning message:
"package 'dplyr' was built under R version 4.5.2"

Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Warning message:
"package 'readr' was built under R version 4.5.2"
Warning message:
"package 'spdep' was built under R version 4.5.2"
Loading required package: spData

Warning message:
"package 'spData' was built under R version 4.5.2"
To access larger datasets in this package, install the spDataLarge
package with: `install.packages('spDataLarge',
repos='https://nowosad.github.io/drat/', type='sou

Running Moran's I for LCBD - zone: All 


Warning message in knn2nb(knn):
"neighbour object has 5 sub-graphs"


Running Moran's I for LCBD - zone: Cold 
Running Moran's I for LCBD - zone: Intermediate 


Warning message in knn2nb(knn):
"neighbour object has 3 sub-graphs"


Running Moran's I for LCBD - zone: Warm 


Warning message in knn2nb(knn):
"neighbour object has 3 sub-graphs"


          Zone N_sites    Moran_I       P_value
1          All    3402 0.20792525 1.369017e-186
2         Cold     670 0.07435279  6.965038e-07
3 Intermediate    2002 0.20627981 1.396442e-111
4         Warm     730 0.15431523  5.448055e-25
🎉 Moran’s I (site-level residuals) for LCBD completed


In [ ]:
# =========================================================
# SES_FDis attribution (NO period split)
# Overall marginal effects
# =========================================================

library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(car)
library(MuMIn)
library(marginaleffects)
library(tibble)

# =========================
# 1) Read data
# =========================
df <- read_csv("D:/NC/Data/rivernet/inputdata/Richness_slope.csv")

df <- df %>%
  mutate(
    SiteID   = as.character(SiteID),
    HYBAS_ID = as.character(HYBAS_ID),
    Protocol = as.character(Protocol),
    zone     = as.character(zone)
  )

# =========================
# 2) Numeric columns
# =========================
num_cols <- c(
  "slope_Fdis",
  "mean_temp","slope_temp","mean_flow",
  "mean_organic","mean_salinity",
  "HFP_mean","elevation"
)

df[num_cols] <- lapply(df[num_cols], as.numeric)

df <- df %>%
  filter(
    complete.cases(
      slope_Fdis,
      mean_temp, slope_temp, mean_flow,
      mean_organic, mean_salinity,
      HFP_mean, elevation,
      Protocol, zone, HYBAS_ID
    )
  )

# =========================
# 3) Z-score predictors
# =========================
vars <- c(
  "mean_temp","slope_temp","mean_flow",
  "mean_organic","mean_salinity",
  "HFP_mean","elevation"
)

for (v in vars) {
  df[[paste0(v, "_z")]] <- scale(df[[v]])[, 1]
}

# =========================
# 4) Model structure (NO period terms)
# =========================
fixed_form <- slope_Fdis ~
  mean_temp_z + slope_temp_z + mean_flow_z +
  mean_organic_z + mean_salinity_z +
  HFP_mean_z + elevation_z

form <- update(
  fixed_form,
  . ~ . + (1 | HYBAS_ID) + (1 | Protocol)
)

# =========================
# 5) VIF helpers
# =========================
zones <- c("All", "Cold", "Intermediate", "Warm")
vif_threshold <- 10
core_vars <- c("mean_temp", "slope_temp")

get_vif_vals <- function(dat) {
  lm_fit <- lm(fixed_form, data = dat)
  vif_raw <- car::vif(lm_fit)
  if (is.matrix(vif_raw)) {
    vif_raw[, "GVIF^(1/(2*Df))"]
  } else {
    vif_raw
  }
}

is_high_vif <- function(v, vif_vals) {
  if (v %in% core_vars) return(FALSE)
  term <- paste0(v, "_z")
  if (!(term %in% names(vif_vals))) return(FALSE)
  isTRUE(vif_vals[term] >= vif_threshold)
}

sig_label <- function(p) {
  if (is.na(p)) "" else if (p < 0.001) "***"
  else if (p < 0.01) "**"
  else if (p < 0.05) "*"
  else ""
}

# =========================
# 6) Run by zone
# =========================
effects_all <- list()
model_stats <- list()
vif_tables  <- list()

for (z in zones) {

  dat <- if (z == "All") df else filter(df, zone == z)
  if (nrow(dat) < 50) next

  cat("Running zone:", z, "\n")

  # -------- VIF --------
  vif_vals <- get_vif_vals(dat)
  vif_tables[[z]] <- data.frame(
    Zone = z,
    Term = names(vif_vals),
    VIF  = as.numeric(vif_vals),
    row.names = NULL
  )

  high_vif <- sapply(vars, is_high_vif, vif_vals = vif_vals)
  names(high_vif) <- vars

  # -------- Model --------
  m <- lmer(
    form, data = dat, REML = TRUE,
    control = lmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
  )

  # -------- Save summary --------
  writeLines(
    capture.output(summary(m)),
    paste0("C:/Users/Lenovo/Modelling of function diversity trends/summary_FDis_", z, ".txt")
  )

  # -------- Fixed effects --------
  coef_tab <- coef(summary(m)) %>%
    as.data.frame() %>%
    rownames_to_column("Term") %>%
    mutate(Zone = z)

  write_csv(
    coef_tab,
    paste0("C:/Users/Lenovo/Modelling of function diversity trends/coef_FDis_", z, ".csv")
  )

  # -------- Random effects --------
  vc_tab <- as.data.frame(VarCorr(m)) %>%
    mutate(Zone = z)

  write_csv(
    vc_tab,
    paste0("C:/Users/Lenovo/Modelling of function diversity trends/varcomp_FDis_", z, ".csv")
  )

  # -------- Fit statistics --------
  r2 <- suppressWarnings(MuMIn::r.squaredGLMM(m))
  model_stats[[z]] <- data.frame(
    Zone = z,
    AIC = AIC(m),
    BIC = BIC(m),
    R2_marginal = r2[1],
    R2_conditional = r2[2],
    row.names = NULL
  )

  # -------- Marginal effects --------
  for (v in vars) {

    if (isTRUE(high_vif[[v]])) {
      effects_all[[length(effects_all) + 1]] <- data.frame(
        Zone = z,
        Factor = v,
        Effect = NA,
        CI95_L = NA,
        CI95_U = NA,
        p = NA,
        sig = "",
        row.names = NULL
      )
      next
    }

    vz <- paste0(v, "_z")

    ame <- avg_slopes(m, variables = vz, newdata = dat)

    effects_all[[length(effects_all) + 1]] <- data.frame(
      Zone = z,
      Factor = v,
      Effect = ame$estimate[1],
      CI95_L = ame$conf.low[1],
      CI95_U = ame$conf.high[1],
      p = ame$p.value[1],
      sig = sig_label(ame$p.value[1]),
      row.names = NULL
    )
  }
}

# =========================
# 7) Output ((Dataset S5))
# =========================
write_csv(bind_rows(effects_all),
          "C:/Users/Lenovo/Modelling of function diversity trends/FDis_MarginalEffects_by_zone_OVERALL.csv")

write_csv(bind_rows(model_stats),
          "C:/Users/Lenovo/Modelling of function diversity trends/Model_fit_stats_FDis.csv")


Rows: 3417 Columns: 34
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (7): SiteID, Protocol, FlowClass, elev_class, HFP_class, geometry, zone
dbl (27): start_year, end_year, n_obs, slope_richness, p_richness, slope_tem...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Running zone: All 


Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning messa

Running zone: Cold 


boundary (singular) fit: see help('isSingular')

Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe 

Running zone: Intermediate 


boundary (singular) fit: see help('isSingular')

Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe 

Running zone: Warm 


boundary (singular) fit: see help('isSingular')

Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe = FALSE)` to silence this warning."
Warning message:
"For this model type, `marginaleffects` only takes into account the uncertainty in fixed-effect parameters. This is often appropriate when `re.form=NA`, but may be surprising to users who set `re.form=NULL` (default) or to some other value. Call `options(marginaleffects_safe 

🎉 完成：SES_FDis mixed models + VIF + avg_slopes (overall)


In [ ]:
library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(ggplot2)
library(patchwork)

# =========================
# 1) Read data
# =========================
df <- read_csv("D:/NC/Data/rivernet/inputdata/Richness_slope.csv")

df <- df %>%
  mutate(
    SiteID   = as.character(SiteID),
    HYBAS_ID = as.character(HYBAS_ID),
    Protocol = as.character(Protocol),
    zone     = as.character(zone)
  )

num_cols <- c(
  "slope_Fdis",
  "mean_temp","slope_temp","mean_flow",
  "mean_organic","mean_salinity",
  "HFP_mean","elevation"
)
df[num_cols] <- lapply(df[num_cols], as.numeric)

df <- df %>%
  filter(
    complete.cases(
      slope_Fdis,
      mean_temp, slope_temp, mean_flow,
      mean_organic, mean_salinity,
      HFP_mean, elevation,
      Protocol, zone, HYBAS_ID
    )
  )

# =========================
# 2) Z-score predictors
# =========================
vars <- c(
  "mean_temp","slope_temp","mean_flow",
  "mean_organic","mean_salinity",
  "HFP_mean","elevation"
)

for (v in vars) {
  df[[paste0(v, "_z")]] <- scale(df[[v]])[, 1]
}

# =========================
# 3) Model formula
# =========================
form <- slope_Fdis ~
  mean_temp_z + slope_temp_z + mean_flow_z +
  mean_organic_z + mean_salinity_z +
  HFP_mean_z + elevation_z +
  (1 | HYBAS_ID) + (1 | Protocol)

zones <- c("All", "Cold", "Intermediate", "Warm")
plots <- list()

# =========================
# 4) Fit & diagnostics
# =========================
for (z in zones) {

  dat <- if (z == "All") df else filter(df, zone == z)
  if (nrow(dat) < 50) next

  m <- lmer(
    form, data = dat, REML = TRUE,
    control = lmerControl(
      optimizer = "bobyqa",
      optCtrl = list(maxfun = 2e5)
    )
  )

  diag_df <- data.frame(
    fitted = fitted(m),
    resid  = resid(m)
  )

  # Residuals vs Fitted
  p1 <- ggplot(diag_df, aes(fitted, resid)) +
    geom_point(size = 0.8, alpha = 0.7) +
    geom_hline(yintercept = 0, linetype = "dashed") +
    labs(
      title = paste(z, ": Residuals vs Fitted"),
      x = "Fitted values",
      y = "Residuals"
    ) +
    theme_bw(base_size = 11)

  # Normal QQ
  p2 <- ggplot(diag_df, aes(sample = resid)) +
    stat_qq(size = 0.8, alpha = 0.8) +
    stat_qq_line(color = "red", linewidth = 0.8) +
    labs(
      title = paste(z, ": Normal Q–Q"),
      x = "Theoretical quantiles",
      y = "Sample quantiles"
    ) +
    theme_bw(base_size = 11)

  plots[[z]] <- p1 + p2
}

# =========================
# 5) Combine & save
# =========================
final_plot <- wrap_plots(plots, ncol = 1)

ggsave(
  "C:/Users/Lenovo/Modelling of function diversity trends/S4.png",
  plot = final_plot,
  width = 10,
  height = 14,
  dpi = 400
)


Rows: 3417 Columns: 34
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (7): SiteID, Protocol, FlowClass, elev_class, HFP_class, geometry, zone
dbl (27): start_year, end_year, n_obs, slope_richness, p_richness, slope_tem...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')



✅ SES_FDis residual diagnostics saved


In [ ]:
library(lme4)
library(lmerTest)
library(dplyr)
library(readr)
library(spdep)

# =========================
# 1) Read data
# =========================
df <- read_csv("D:/NC/Data/rivernet/inputdata/Richness_slope.csv")

df <- df %>%
  mutate(
    SiteID   = as.character(SiteID),
    HYBAS_ID = as.character(HYBAS_ID),
    Protocol = as.character(Protocol),
    zone     = as.character(zone)
  )

# =========================
# 2) Numeric columns + filter
# =========================
num_cols <- c(
  "slope_Fdis",
  "mean_temp","slope_temp","mean_flow",
  "mean_organic","mean_salinity",
  "HFP_mean","elevation",
  "Longitude","Latitude"
)

df[num_cols] <- lapply(df[num_cols], as.numeric)

df <- df %>%
  filter(
    complete.cases(
      slope_Fdis,
      mean_temp, slope_temp, mean_flow,
      mean_organic, mean_salinity,
      HFP_mean, elevation,
      Protocol, zone, HYBAS_ID, SiteID,
      Longitude, Latitude
    )
  )

# =========================
# 3) Z-score predictors
# =========================
scale_cols <- c(
  "mean_temp","slope_temp","mean_flow",
  "mean_organic","mean_salinity",
  "HFP_mean","elevation"
)

for (v in scale_cols) {
  df[[paste0(v, "_z")]] <- scale(df[[v]])[, 1]
}

# =========================
# 4) Mixed model (align with your LCBD style)
# =========================
form <- slope_Fdis ~
  mean_temp_z + slope_temp_z + mean_flow_z +
  mean_organic_z + mean_salinity_z +
  HFP_mean_z + elevation_z +
  (1 | HYBAS_ID) + (1 | Protocol)

zones <- c("All","Cold","Intermediate","Warm")
out <- list()

# =========================
# 5) Moran’s I by zone (site-level residuals)
# =========================
for (z in zones) {

  dat <- if (z == "All") df else filter(df, zone == z)
  if (nrow(dat) < 50) next

  cat("Running Moran's I for zone:", z, "\n")

  m <- lmer(
    form, data = dat, REML = TRUE,
    control = lmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
  )

  # align residuals with used rows
  mf <- model.frame(m)
  mf$resid <- resid(m)
  dat_used <- dat[as.numeric(rownames(mf)), ]

  mf$SiteID    <- dat_used$SiteID
  mf$Longitude <- dat_used$Longitude
  mf$Latitude  <- dat_used$Latitude

  # aggregate to SiteID (site-level residuals)
  site_res <- mf %>%
    group_by(SiteID) %>%
    summarise(
      resid = mean(resid, na.rm = TRUE),
      Longitude = first(Longitude),
      Latitude  = first(Latitude),
      .groups = "drop"
    ) %>%
    filter(complete.cases(resid, Longitude, Latitude))

  coords <- as.matrix(site_res[, c("Longitude", "Latitude")])

  # kNN weights (same as your LCBD workflow)
  knn <- knearneigh(coords, k = 10)
  nb  <- knn2nb(knn)
  lw  <- nb2listw(nb, style = "W", zero.policy = TRUE)

  mi <- moran.test(
    site_res$resid,
    lw,
    randomisation = TRUE,
    zero.policy = TRUE
  )

  out[[z]] <- data.frame(
    Zone    = z,
    N_sites = nrow(site_res),
    Moran_I = unname(mi$estimate["Moran I statistic"]),
    P_value = mi$p.value,
    row.names = NULL
  )
}

df_moran <- bind_rows(out)

print(df_moran)
#((Dataset S5))
write_csv(
  df_moran,
  "C:/Users/Lenovo/Modelling of function diversity trends/MoranI_residuals_by_zone_SITE_SES_FDis.csv"
)


Rows: 3417 Columns: 34
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (7): SiteID, Protocol, FlowClass, elev_class, HFP_class, geometry, zone
dbl (27): start_year, end_year, n_obs, slope_richness, p_richness, slope_tem...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Running Moran's I for zone: All 


Warning message in knn2nb(knn):
"neighbour object has 6 sub-graphs"


Running Moran's I for zone: Cold 


boundary (singular) fit: see help('isSingular')



Running Moran's I for zone: Intermediate 


boundary (singular) fit: see help('isSingular')

Warning message in knn2nb(knn):
"neighbour object has 4 sub-graphs"


Running Moran's I for zone: Warm 


boundary (singular) fit: see help('isSingular')

Warning message in knn2nb(knn):
"neighbour object has 3 sub-graphs"


          Zone N_sites       Moran_I     P_value
1          All    2757  0.0192806652 0.007038252
2         Cold     362 -0.0265515777 0.867766628
3 Intermediate    1733  0.0183516693 0.029422510
4         Warm     662 -0.0003945008 0.472647184
🎉 Moran’s I (site-level residuals) for SES_FDis completed
